# nnUNet v2 — Liver & Liver Tumor Segmentation (Kaggle / NPY Dataset)
**Model**: nnUNet v2 (MIC-DKFZ)  
**Dataset**: `zeynepzelk/task03-liver-npy-dataset` (pre-converted `.npy` format)  
**Runtime required**: GPU (T4 16 GB = 2D training; P100/A100 = 3D training)  
**Estimated time**:
- Mode A (inference only with pretrained weights): ~20 min  
- Mode B (train from scratch, 2D, 1 fold, 100 epochs): ~4–6 h on T4  

**Labels**: 0 = background · 1 = liver · 2 = tumor  

---
### What this notebook covers
1. Environment setup and GPU check  
2. NPY dataset loading from Kaggle input path  
3. NIfTI conversion & nnUNet-format dataset preparation  
4. Dataset fingerprinting + automatic pipeline configuration  
5. Training (2D config, fold 0, 100 epochs)  
6. Inference on held-out validation cases  
7. Evaluation: Dice, HD95, Jaccard — separately for liver and tumor  
8. Visualisation: segmentation overlays, metric tables, training curves  


## Cell 1 — Hardware Check

In [12]:
import subprocess, sys, os, shutil

# Check GPU — nvidia-smi may live in /usr/bin or /usr/local/bin on Kaggle
nvidia_smi = shutil.which('nvidia-smi') or '/usr/bin/nvidia-smi'
try:
    gpu_info = subprocess.run([nvidia_smi], capture_output=True, text=True)
    if gpu_info.returncode == 0:
        print(gpu_info.stdout)
    else:
        print("⚠️  nvidia-smi found but returned an error — GPU may not be active")
        print(gpu_info.stderr)
except FileNotFoundError:
    # Fall back to torch to confirm GPU availability
    import torch
    if torch.cuda.is_available():
        print(f"✅ GPU available via PyTorch: {torch.cuda.get_device_name(0)}")
        print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("⚠️  No GPU detected. Enable GPU in Kaggle: Settings → Accelerator → GPU T4 x2")

# Check RAM
import psutil
ram = psutil.virtual_memory()
print(f"\nRAM: {ram.total / 1e9:.1f} GB total | {ram.available / 1e9:.1f} GB available")

# Check disk
disk = psutil.disk_usage('/')
print(f"Disk: {disk.total / 1e9:.1f} GB total | {disk.free / 1e9:.1f} GB free")
print("\n✅ Need at least: 16 GB VRAM | 20 GB disk | 12 GB RAM")

# Verify Kaggle dataset is mounted
DATASET_ROOT = '/kaggle/input/datasets/zeynepzelk/task03-liver-npy-dataset'
for sub in ['image', 'liverMask', 'tumorMask']:
    path = os.path.join(DATASET_ROOT, sub)
    n = len(os.listdir(path)) if os.path.isdir(path) else 0
    print(f"  {sub}/  → {n} files")
print("✅ Dataset found" if os.path.isdir(DATASET_ROOT) else "❌ Dataset NOT found — add it via Add Data")

⚠️  No GPU detected. Enable GPU in Kaggle: Settings → Accelerator → GPU T4 x2

RAM: 33.7 GB total | 32.2 GB available
Disk: 8656.9 GB total | 1310.3 GB free

✅ Need at least: 16 GB VRAM | 20 GB disk | 12 GB RAM
  image/  → 131 files
  liverMask/  → 130 files
  tumorMask/  → 130 files
✅ Dataset found


## Cell 2 — Install nnUNet v2 and Dependencies
Kaggle already has PyTorch; we only need nnUNet and a few extras.  
This takes ~2–3 minutes. Run once per session.

In [13]:
import os, glob, json, shutil
import numpy as np
import nibabel as nib
from tqdm.notebook import tqdm

# ── Paths ────────────────────────────────────────────────────────────────────
DATASET_ROOT = '/kaggle/input/datasets/zeynepzelk/task03-liver-npy-dataset'
IMG_DIR      = os.path.join(DATASET_ROOT, 'image')
LIVER_DIR    = os.path.join(DATASET_ROOT, 'liverMask')
TUMOR_DIR    = os.path.join(DATASET_ROOT, 'tumorMask')

DATASET_NAME = 'Dataset003_Liver'
DATASET_PATH = os.path.join(RAW_DIR, DATASET_NAME)
IMAGES_TR    = os.path.join(DATASET_PATH, 'imagesTr')
LABELS_TR    = os.path.join(DATASET_PATH, 'labelsTr')
IMAGES_TS    = os.path.join(DATASET_PATH, 'imagesTs')

for d in [IMAGES_TR, LABELS_TR, IMAGES_TS]:
    os.makedirs(d, exist_ok=True)

# ── Discover cases ───────────────────────────────────────────────────────────
img_files = sorted(glob.glob(os.path.join(IMG_DIR, '*.npy')))
print(f"Found {len(img_files)} cases")

# ── Train / Val split ────────────────────────────────────────────────────────
np.random.seed(42)
indices = np.arange(len(img_files))
np.random.shuffle(indices)
val_count = max(1, int(0.10 * len(img_files)))
train_cases = [img_files[i] for i in indices[val_count:]]
val_cases   = [img_files[i] for i in indices[:val_count]]

def npy_to_nifti(img_path, liver_path, tumor_path, out_img, out_lbl):
    """Load .npy, merge masks, and standardize to 3D (H, W, Slices)."""
    vol   = np.load(img_path).astype(np.float32)
    liver = np.load(liver_path).astype(np.uint8)
    tumor = np.load(tumor_path).astype(np.uint8)

    combined = np.zeros_like(liver, dtype=np.uint8)
    combined[liver == 1] = 1
    combined[tumor == 2] = 2 # Tumor overwrites liver labels in overlaps

    # FIX: Transpose from (Slices, H, W) to (H, W, Slices) 
    # nnUNet and Nibabel expect spatial dimensions to be in the first 3 axes.
    # If we don't transpose, nnUNet thinks 'Slices' is a channel/modality.
    if vol.ndim == 3:
        vol = vol.transpose(1, 2, 0)
    if combined.ndim == 3:
        combined = combined.transpose(1, 2, 0)

    # Ensure they are 3D (X, Y, Z). Do NOT add np.newaxis here.
    affine = np.eye(4)
    
    # Final check: Ensure spatial dimensions match exactly
    if vol.shape != combined.shape:
        # If image is a single slice but mask is a volume, match the image to the mask
        if vol.shape[2] == 1 and combined.shape[2] > 1:
            vol = np.repeat(vol, combined.shape[2], axis=2)

    nib.save(nib.Nifti1Image(vol, affine), out_img)
    nib.save(nib.Nifti1Image(combined, affine), out_lbl)

def get_pair_paths(img_path):
    base = os.path.basename(img_path).replace('_img.npy', '')
    return base, os.path.join(LIVER_DIR, base + '_liverMask.npy'), os.path.join(TUMOR_DIR, base + '_tumorMask.npy')

# ── Execution ────────────────────────────────────────────────────────────────
print("\nConverting training cases...")
train_ids = []
for p in tqdm(train_cases):
    base, lp, tp = get_pair_paths(p)
    if os.path.exists(lp) and os.path.exists(tp):
        npy_to_nifti(p, lp, tp, os.path.join(IMAGES_TR, f"{base}_0000.nii.gz"), os.path.join(LABELS_TR, f"{base}.nii.gz"))
        train_ids.append(base)

print("Converting validation cases...")
val_ids = []
for p in tqdm(val_cases):
    base, lp, tp = get_pair_paths(p)
    if os.path.exists(lp) and os.path.exists(tp):
        npy_to_nifti(p, lp, tp, os.path.join(IMAGES_TS, f"{base}_0000.nii.gz"), os.path.join(LABELS_TR, f"{base}.nii.gz"))
        val_ids.append(base)

print(f"✅ Conversion complete: {len(train_ids)} train, {len(val_ids)} val")

Found 131 cases

Converting training cases...


  0%|          | 0/118 [00:00<?, ?it/s]

Converting validation cases...


  0%|          | 0/13 [00:00<?, ?it/s]

✅ Conversion complete: 117 train, 13 val


## Cell 3 — Set nnUNet Environment Variables
All paths live under `/kaggle/working/` which is the writable workspace.

In [14]:
import os

WORKSPACE      = '/kaggle/working'
RAW_DIR        = os.path.join(WORKSPACE, 'nnUNet_raw')
PREPROCESSED   = os.path.join(WORKSPACE, 'nnUNet_preprocessed')
RESULTS_DIR    = os.path.join(WORKSPACE, 'nnUNet_results')

for d in [RAW_DIR, PREPROCESSED, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

os.environ['nnUNet_raw']         = RAW_DIR
os.environ['nnUNet_preprocessed'] = PREPROCESSED
os.environ['nnUNet_results']     = RESULTS_DIR

print(f"nnUNet_raw         → {RAW_DIR}")
print(f"nnUNet_preprocessed → {PREPROCESSED}")
print(f"nnUNet_results     → {RESULTS_DIR}")
print("✅ Environment variables set")

nnUNet_raw         → /kaggle/working/nnUNet_raw
nnUNet_preprocessed → /kaggle/working/nnUNet_preprocessed
nnUNet_results     → /kaggle/working/nnUNet_results
✅ Environment variables set


## Cell 4 — Load NPY Dataset and Convert to NIfTI (nnUNet Format)

**Dataset layout** at `/kaggle/input/datasets/zeynepzelk/task03-liver-npy-dataset`:
```
image/          liver_XX_img.npy        — normalized 3-D CT volume
liverMask/      liver_XX_liverMask.npy  — 0 = bg, 1 = liver
tumorMask/      liver_XX_tumorMask.npy  — 0 = bg, 2 = tumor
```

We merge liver + tumor masks into a single multi-class label:  
`0` = background · `1` = liver · `2` = tumor  
then save each case as `.nii.gz` under `Dataset003_Liver/`.

In [21]:
import os, glob, json, shutil
import numpy as np
import nibabel as nib
from tqdm.notebook import tqdm

# ── Paths ────────────────────────────────────────────────────────────────────
DATASET_ROOT = '/kaggle/input/datasets/zeynepzelk/task03-liver-npy-dataset'
IMG_DIR      = os.path.join(DATASET_ROOT, 'image')
LIVER_DIR    = os.path.join(DATASET_ROOT, 'liverMask')
TUMOR_DIR    = os.path.join(DATASET_ROOT, 'tumorMask')

DATASET_NAME = 'Dataset003_Liver'
DATASET_PATH = os.path.join(RAW_DIR, DATASET_NAME)
IMAGES_TR    = os.path.join(DATASET_PATH, 'imagesTr')
LABELS_TR    = os.path.join(DATASET_PATH, 'labelsTr')
IMAGES_TS    = os.path.join(DATASET_PATH, 'imagesTs')

for d in [IMAGES_TR, LABELS_TR, IMAGES_TS]:
    os.makedirs(d, exist_ok=True)

# ── Discover all cases via image folder ──────────────────────────────────────
img_files = sorted(glob.glob(os.path.join(IMG_DIR, '*.npy')))
print(f"Found {len(img_files)} cases in the dataset")

# ── Train / Val split (90 % / 10 %) ─────────────────────────────────────────
np.random.seed(42)
indices     = np.arange(len(img_files))
np.random.shuffle(indices)
val_count   = max(1, int(0.10 * len(img_files)))
val_idx     = set(indices[:val_count])
train_cases = [img_files[i] for i in indices[val_count:]]
val_cases   = [img_files[i] for i in indices[:val_count]]

print(f"Train: {len(train_cases)} | Val (held-out): {len(val_cases)}")

def npy_to_nifti(img_path, liver_path, tumor_path, out_img, out_lbl):
    """
    Load .npy arrays (shape: D, H, W = slices, 512, 512),
    merge liver + tumor masks, and save as .nii.gz in NIfTI/ITK
    convention (W, H, D) so that nnUNet's SimpleITK reader sees the
    correct 3-D shape and both image & label share the same affine.

    Fixes:
      - Shape mismatch (512,512,1) vs (75,512,512): caused by adding
        a channel dim with vol[np.newaxis] before transposing.
      - Affine broadcast error (4,) vs (3,): caused by image and label
        having different NIfTI header affines.
    """
    vol   = np.load(img_path).astype(np.float32)   # (D, H, W)
    liver = np.load(liver_path).astype(np.uint8)   # (D, H, W)
    tumor = np.load(tumor_path).astype(np.uint8)   # (D, H, W)

    # Drop any accidental leading singleton dimension
    if vol.ndim   == 4 and vol.shape[0]   == 1: vol   = vol[0]
    if liver.ndim == 4 and liver.shape[0] == 1: liver = liver[0]
    if tumor.ndim == 4 and tumor.shape[0] == 1: tumor = tumor[0]

    assert vol.ndim == 3,   f'Image must be 3-D, got {vol.shape}'
    assert liver.ndim == 3, f'liverMask must be 3-D, got {liver.shape}'
    assert tumor.ndim == 3, f'tumorMask must be 3-D, got {tumor.shape}'

    # Merge: 0=bg, 1=liver, 2=tumor
    combined = np.zeros_like(liver, dtype=np.uint8)
    combined[liver == 1] = 1
    combined[tumor == 2] = 2  # tumor overwrites liver in overlap

    # Transpose (D, H, W) -> (W, H, D) for NIfTI/ITK x-y-z order
    # nnUNet uses SimpleITK which expects (W, H, D) in memory.
    # NOT adding a channel dim here — the volume is purely 3-D.
    vol_nii = np.transpose(vol,      (2, 1, 0))  # (W, H, D)
    seg_nii = np.transpose(combined, (2, 1, 0))  # (W, H, D)

    # Shared identity affine (1 mm isotropic).
    # Image and label MUST share the same affine object so that
    # nnUNet's verify_dataset_integrity np.allclose(origin_img, origin_seg)
    # comparison never fails with a broadcast shape mismatch.
    affine = np.eye(4, dtype=np.float64)

    nib.save(nib.Nifti1Image(vol_nii, affine), out_img)
    nib.save(nib.Nifti1Image(seg_nii, affine), out_lbl)

def get_pair_paths(img_path):
    """Derive liverMask and tumorMask paths from image path."""
    base = os.path.basename(img_path).replace('_img.npy', '')
    liver_path = os.path.join(LIVER_DIR, base + '_liverMask.npy')
    tumor_path = os.path.join(TUMOR_DIR, base + '_tumorMask.npy')
    return base, liver_path, tumor_path

# ── Convert training cases ────────────────────────────────────────────────────
print("\nConverting training cases to NIfTI ...")
train_ids = []
for img_path in tqdm(train_cases):
    base, liver_path, tumor_path = get_pair_paths(img_path)
    if not (os.path.exists(liver_path) and os.path.exists(tumor_path)):
        print(f"  ⚠️  Missing mask for {base}, skipping")
        continue
    case_id = base  # e.g. liver_00
    out_img = os.path.join(IMAGES_TR, f"{case_id}_0000.nii.gz")
    out_lbl = os.path.join(LABELS_TR, f"{case_id}.nii.gz")
    npy_to_nifti(img_path, liver_path, tumor_path, out_img, out_lbl)
    train_ids.append(case_id)

# ── Convert validation cases (save to imagesTs for inference) ────────────────
print("\nConverting validation cases ...")
val_ids = []
for img_path in tqdm(val_cases):
    base, liver_path, tumor_path = get_pair_paths(img_path)
    if not (os.path.exists(liver_path) and os.path.exists(tumor_path)):
        continue
    case_id = base
    out_img = os.path.join(IMAGES_TS, f"{case_id}_0000.nii.gz")
    # Also store GT labels for evaluation
    out_lbl = os.path.join(LABELS_TR, f"{case_id}.nii.gz")
    npy_to_nifti(img_path, liver_path, tumor_path, out_img, out_lbl)
    val_ids.append(case_id)

print(f"\n✅ {len(train_ids)} training + {len(val_ids)} validation cases converted")

Found 131 cases in the dataset
Train: 118 | Val (held-out): 13

Converting training cases to NIfTI ...


  0%|          | 0/118 [00:00<?, ?it/s]

  ⚠️  Missing mask for liver_98, skipping

Converting validation cases ...


  0%|          | 0/13 [00:00<?, ?it/s]


✅ 117 training + 13 validation cases converted


## Cell 5 — Write dataset.json

In [22]:
import json

dataset_json = {
    "channel_names": {"0": "CT"},
    "labels": {
        "background": 0,
        "liver":      1,
        "tumor":      2
    },
    "numTraining": len(train_ids),
    "file_ending": ".nii.gz",
    "name": "Dataset003_Liver",
    "description": "MSD Task03 Liver in NPY → NIfTI via Kaggle dataset zeynepzelk/task03-liver-npy-dataset",
    "reference": "http://medicaldecathlon.com",
    "licence": "CC-BY-SA 4.0",
    "release": "1.0",
    "tensorImageSize": "3D"
}

json_path = os.path.join(DATASET_PATH, 'dataset.json')
with open(json_path, 'w') as f:
    json.dump(dataset_json, f, indent=2)

print(f"✅ dataset.json written to {json_path}")
print(f"   Training cases : {dataset_json['numTraining']}")
print(f"   Labels         : {dataset_json['labels']}")

✅ dataset.json written to /kaggle/working/nnUNet_raw/Dataset003_Liver/dataset.json
   Training cases : 117
   Labels         : {'background': 0, 'liver': 1, 'tumor': 2}


## Cell 6 — Dataset Fingerprinting & Automatic Pipeline Configuration
nnUNet analyses voxel spacing, intensity statistics, and dataset size to decide
the optimal patch size, batch size, and network architecture automatically.

In [24]:
import subprocess

env = os.environ.copy()
DATASET_ID = 3

print("Running nnUNetv2_plan_and_preprocess ...")
result = subprocess.run(
    ['nnUNetv2_plan_and_preprocess',
     '-d', str(DATASET_ID),
     '--verify_dataset_integrity',
     '-c', '2d'],
    env=env, text=True, capture_output=False
)

if result.returncode == 0:
    print("✅ Fingerprinting and preprocessing complete")
else:
    print(f"⚠️  Preprocessing exited with code {result.returncode}")

Running nnUNetv2_plan_and_preprocess ...
Fingerprint extraction...
Dataset003_Liver
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [1.03 1.03 1.03]. 
Current patch size: (np.int64(256), np.int64(224), np.int64(40)). 
Current median shape: [497.08737864 347.57281553  72.81553398]
Attempting to find 3d_lowres config. 
Current spacing: [1.0609 1.0609 1.0609]. 
Current patch size: (np.int64(256), np.int64(224), np.int64(40)). 
Current median shape: [4

✅ Fingerprinting and preprocessing complete


## Cell 7 — Training (2D Configuration, Fold 0, 100 Epochs)
Switch `NUM_EPOCHS` to `1000` for production-quality results (+2–3 Dice).  
Training on a T4 takes ~4–6 h at 100 epochs.

In [29]:
import subprocess, os

DATASET_ID    = 3
CONFIGURATION = '2d'
FOLD          = 0
NUM_EPOCHS    = 100   # increase to 1000 for best results

env = os.environ.copy()
# nnUNet v2 reads epoch count from this environment variable
env['nnUNet_n_proc_DA'] = '8'
env['nnUNet_def_n_proc'] = '8'

# Patch the trainer to use our epoch count via a quick monkey-patch file
trainer_override = f"""
import nnunetv2.training.nnUNetTrainer.nnUNetTrainer as _m
_m.nnUNetTrainer.max_num_epochs = {NUM_EPOCHS}
"""
override_path = '/kaggle/working/epoch_patch.py'
with open(override_path, 'w') as f:
    f.write(trainer_override)

env['PYTHONSTARTUP'] = override_path

print(f"Starting training: Dataset {DATASET_ID} | {CONFIGURATION} | fold {FOLD} | {NUM_EPOCHS} epochs")
result = subprocess.run(
    ['nnUNetv2_train',
     str(DATASET_ID),
     CONFIGURATION,
     str(FOLD),
     '--npz'],
    env=env, text=True, capture_output=False
)

if result.returncode == 0:
    print("\n✅ Training complete")
else:
    print(f"\n⚠️  Training exited with code {result.returncode}")

Starting training: Dataset 3 | 2d | fold 0 | 100 epochs



############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################




⚠️  Training exited with code 1


## Cell 8 — Inference on Held-Out Validation Cases
Runs the trained model on the validation images saved under `imagesTs/`.

In [26]:
import subprocess, os

DATASET_ID    = 3
CONFIGURATION = '2d'
FOLD          = 0

INFER_IN  = os.path.join(RAW_DIR, 'Dataset003_Liver', 'imagesTs')
INFER_OUT = os.path.join(WORKSPACE, 'predictions')
os.makedirs(INFER_OUT, exist_ok=True)

env = os.environ.copy()

print(f"Running inference on {len(os.listdir(INFER_IN))} validation volumes ...")
result = subprocess.run(
    ['nnUNetv2_predict',
     '-i', INFER_IN,
     '-o', INFER_OUT,
     '-d', str(DATASET_ID),
     '-c', CONFIGURATION,
     '-f', str(FOLD),
     '-chk', 'checkpoint_best.pth',
     '--save_probabilities'],
    env=env, text=True, capture_output=False
)

if result.returncode == 0:
    preds = [f for f in os.listdir(INFER_OUT) if f.endswith('.nii.gz')]
    print(f"\n✅ Inference complete — {len(preds)} prediction files in {INFER_OUT}")
else:
    print(f"\n⚠️  Inference failed (exit code {result.returncode})")

Running inference on 13 validation volumes ...



#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################


⚠️  Inference failed (exit code 1)


## Cell 9 — Evaluation: Dice, HD95, Jaccard
Computes per-case metrics for liver (label=1) and tumor (label=2),
then prints a summary table.

In [27]:
import os, glob
import numpy as np
import nibabel as nib
import pandas as pd
from tqdm.notebook import tqdm

try:
    import surface_distance as sd
    HAS_SD = True
except ImportError:
    HAS_SD = False
    print("⚠️  surface-distance not available — HD95 will be skipped")

LABELS_DIR = os.path.join(RAW_DIR, 'Dataset003_Liver', 'labelsTr')
PRED_DIR   = os.path.join(WORKSPACE, 'predictions')

def dice(pred, gt, label):
    p = (pred == label).astype(bool)
    g = (gt   == label).astype(bool)
    if g.sum() == 0 and p.sum() == 0:
        return 1.0
    if g.sum() == 0 or p.sum() == 0:
        return 0.0
    return 2 * (p & g).sum() / (p.sum() + g.sum())

def jaccard(pred, gt, label):
    p = (pred == label).astype(bool)
    g = (gt   == label).astype(bool)
    if g.sum() == 0 and p.sum() == 0:
        return 1.0
    union = (p | g).sum()
    return (p & g).sum() / union if union > 0 else 0.0

def hd95(pred, gt, label, spacing=(1.0, 1.0, 1.0)):
    if not HAS_SD:
        return np.nan
    p = (pred == label).astype(bool)
    g = (gt   == label).astype(bool)
    if g.sum() == 0 and p.sum() == 0:
        return 0.0
    if g.sum() == 0 or p.sum() == 0:
        return np.nan
    surf = sd.compute_surface_distances(g, p, spacing_mm=spacing)
    return sd.compute_robust_hausdorff(surf, 95)

pred_files = sorted(glob.glob(os.path.join(PRED_DIR, '*.nii.gz')))
print(f"Evaluating {len(pred_files)} cases ...")

records = []
for pred_path in tqdm(pred_files):
    case_id = os.path.basename(pred_path).replace('_0000.nii.gz', '').replace('.nii.gz', '')
    gt_path = os.path.join(LABELS_DIR, f"{case_id}.nii.gz")
    if not os.path.exists(gt_path):
        print(f"  ⚠️  GT not found for {case_id}")
        continue

    pred_vol = nib.load(pred_path).get_fdata().astype(np.uint8)
    gt_vol   = nib.load(gt_path).get_fdata().astype(np.uint8)

    # squeeze extra channel dim if present
    if pred_vol.ndim == 4: pred_vol = pred_vol[0]
    if gt_vol.ndim   == 4: gt_vol   = gt_vol[0]

    records.append({
        'case':         case_id,
        'dice_liver':   dice(pred_vol, gt_vol, 1),
        'dice_tumor':   dice(pred_vol, gt_vol, 2),
        'jacc_liver':   jaccard(pred_vol, gt_vol, 1),
        'jacc_tumor':   jaccard(pred_vol, gt_vol, 2),
        'hd95_liver':   hd95(pred_vol, gt_vol, 1),
        'hd95_tumor':   hd95(pred_vol, gt_vol, 2),
    })

df = pd.DataFrame(records)

# ── Per-case table ────────────────────────────────────────────────────────────
print("\n📊 Per-case metrics:")
display(df.round(4))

# ── Summary statistics ────────────────────────────────────────────────────────
summary = df.drop(columns='case').agg(['mean', 'std', 'min', 'max'])
print("\n📊 Aggregate metrics (mean ± std):")
display(summary.round(4))

# Save to CSV
df.to_csv(os.path.join(WORKSPACE, 'metrics_per_case.csv'), index=False)
summary.to_csv(os.path.join(WORKSPACE, 'metrics_summary.csv'))
print(f"\n✅ Metrics saved to {WORKSPACE}/metrics_*.csv")

Evaluating 0 cases ...


0it [00:00, ?it/s]


📊 Per-case metrics:


""


KeyError: "['case'] not found in axis"

## Cell 10 — Plot Metric Tables and Bar Charts

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('nnUNet v2 — Liver & Tumor Segmentation Metrics (Validation Set)',
             fontsize=14, fontweight='bold', y=1.02)

metrics = [
    ('Dice Score',    ['dice_liver',  'dice_tumor'],  'Dice',    (0, 1)),
    ('Jaccard Score', ['jacc_liver',  'jacc_tumor'],  'Jaccard', (0, 1)),
    ('HD95 (mm)',     ['hd95_liver',  'hd95_tumor'],  'mm',      None),
]

colors = ['#2196F3', '#F44336']   # blue = liver, red = tumor

for ax, (title, cols, ylabel, ylim) in zip(axes, metrics):
    means = [df[c].mean() for c in cols]
    stds  = [df[c].std()  for c in cols]
    x     = np.arange(len(cols))

    bars = ax.bar(x, means, yerr=stds, capsize=8,
                  color=colors, edgecolor='black', linewidth=0.8,
                  error_kw={'elinewidth': 1.5, 'ecolor': '#333'})

    for bar, mean, std in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + std + 0.01,
                f"{mean:.3f}", ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_xticks(x)
    ax.set_xticklabels(['Liver', 'Tumor'], fontsize=11)
    if ylim:
        ax.set_ylim(*ylim)
    ax.grid(axis='y', alpha=0.35)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, 'metrics_bar.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Bar chart saved")

## Cell 11 — Box Plots: Per-Case Distribution of Metrics

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Per-Case Metric Distribution', fontsize=14, fontweight='bold')

bp_data = [
    ('Dice',    [df['dice_liver'].dropna(), df['dice_tumor'].dropna()]),
    ('Jaccard', [df['jacc_liver'].dropna(), df['jacc_tumor'].dropna()]),
    ('HD95 (mm)', [df['hd95_liver'].dropna(), df['hd95_tumor'].dropna()]),
]

for ax, (ylabel, data) in zip(axes, bp_data):
    bp = ax.boxplot(data, labels=['Liver', 'Tumor'], patch_artist=True,
                    medianprops={'color': 'black', 'linewidth': 2})
    for patch, color in zip(bp['boxes'], ['#2196F3', '#F44336']):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel, fontweight='bold')
    ax.grid(axis='y', alpha=0.35)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, 'metrics_boxplot.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Box plot saved")

## Cell 12 — Training Curves (Loss & Dice)
nnUNet saves `progress.png` automatically. If it exists we show it;
otherwise we parse `training_log.txt`.

In [ ]:
import matplotlib.pyplot as plt
import os, re

RESULTS_DIR   = os.environ['nnUNet_results']
DATASET_ID    = 3
CONFIGURATION = '2d'
FOLD          = 0

log_dir = os.path.join(
    RESULTS_DIR,
    f"Dataset{DATASET_ID:03d}_Liver",
    'nnUNetTrainer__nnUNetPlans__2d',
    f'fold_{FOLD}'
)

progress_file = os.path.join(log_dir, 'progress.png')

if os.path.exists(progress_file):
    from PIL import Image
    img = Image.open(progress_file)
    plt.figure(figsize=(14, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title('nnUNet Training Progress (auto-generated by nnUNet)', fontsize=12)
    plt.tight_layout()
    plt.show()
    print("✅ progress.png displayed")
else:
    log_txt = os.path.join(log_dir, 'training_log.txt')
    if os.path.exists(log_txt):
        epochs, train_loss, val_dice = [], [], []
        with open(log_txt) as f:
            for line in f:
                # Parse epoch and train loss
                m = re.search(r'Epoch\s+(\d+).*train_loss\s+([\d.eE+\-]+)', line)
                if m:
                    epochs.append(int(m.group(1)))
                    train_loss.append(float(m.group(2)))
                # Parse validation pseudo-dice
                m2 = re.search(r'pseudo_dice.*?([\d.]+)', line)
                if m2:
                    val_dice.append(float(m2.group(1)))

        if epochs:
            fig, axes = plt.subplots(1, 2, figsize=(14, 4))
            fig.suptitle('nnUNet v2 Training Curves', fontsize=13, fontweight='bold')

            axes[0].plot(epochs, train_loss, 'b-', linewidth=1.5, label='Train loss')
            axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
            axes[0].set_title('Training Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
            axes[0].spines[['top', 'right']].set_visible(False)

            if val_dice:
                ep_dice = range(len(val_dice))
                axes[1].plot(ep_dice, val_dice, 'g-', linewidth=1.5, label='Val Dice (pseudo)')
                axes[1].axhline(max(val_dice), color='r', linestyle='--', alpha=0.6,
                                label=f'Best: {max(val_dice):.4f}')
                axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Dice')
                axes[1].set_title('Validation Dice'); axes[1].legend(); axes[1].grid(alpha=0.3)
                axes[1].spines[['top', 'right']].set_visible(False)

            plt.tight_layout()
            plt.savefig(os.path.join(WORKSPACE, 'training_curves.png'), dpi=150, bbox_inches='tight')
            plt.show()
            print("✅ Training curves plotted and saved")
        else:
            print(f"⚠️  Log exists but no parseable data yet. Training may still be running.\n   Log at: {log_txt}")
    else:
        print(f"⚠️  No training log found at {log_dir}")
        print("    Run training first (Cell 7).")

## Cell 13 — Segmentation Overlay Visualisation
Displays axial slices with colour-coded prediction overlay:  
🔵 **Blue tint** = predicted liver · 🟢 **Green tint** = predicted tumor

In [ ]:
import os, glob
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

PRED_DIR   = os.path.join(WORKSPACE, 'predictions')
LABELS_DIR = os.path.join(RAW_DIR, 'Dataset003_Liver', 'labelsTr')
IMAGES_TS  = os.path.join(RAW_DIR, 'Dataset003_Liver', 'imagesTs')

pred_files = sorted(glob.glob(os.path.join(PRED_DIR, '*.nii.gz')))[:3]  # show up to 3 cases

for pred_path in pred_files:
    case_id = os.path.basename(pred_path).replace('_0000.nii.gz', '').replace('.nii.gz', '')
    ct_path = os.path.join(IMAGES_TS, f"{case_id}_0000.nii.gz")
    gt_path = os.path.join(LABELS_DIR, f"{case_id}.nii.gz")

    if not (os.path.exists(ct_path) and os.path.exists(gt_path)):
        print(f"⚠️  Missing CT or GT for {case_id}")
        continue

    ct   = nib.load(ct_path).get_fdata().astype(np.float32)
    pred = nib.load(pred_path).get_fdata().astype(np.uint8)
    gt   = nib.load(gt_path).get_fdata().astype(np.uint8)

    if ct.ndim == 4:   ct   = ct[0]
    if pred.ndim == 4: pred = pred[0]
    if gt.ndim == 4:   gt   = gt[0]

    # Pick 5 representative axial slices where liver or tumor is present
    liver_slices = np.where(pred.sum(axis=(0, 1)) > 0)[0]
    if len(liver_slices) == 0:
        liver_slices = np.linspace(0, ct.shape[2]-1, 5, dtype=int)
    step   = max(1, len(liver_slices) // 5)
    slices = liver_slices[::step][:5]

    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    fig.suptitle(f'Case: {case_id}  |  Top: Pred overlay  |  Bottom: GT overlay',
                 fontsize=12, fontweight='bold')

    def make_overlay(ct_sl, seg_sl):
        ct_n = (ct_sl - ct_sl.min()) / (ct_sl.max() - ct_sl.min() + 1e-8)
        rgb  = np.stack([ct_n, ct_n, ct_n], axis=-1).copy()
        rgb[seg_sl == 1, 2] = np.clip(rgb[seg_sl == 1, 2] + 0.45, 0, 1)  # blue = liver
        rgb[seg_sl == 2, 1] = np.clip(rgb[seg_sl == 2, 1] + 0.55, 0, 1)  # green = tumor
        return rgb

    legend_handles = [
        Patch(facecolor=(0, 0, 0.6), label='Liver (pred/gt)'),
        Patch(facecolor=(0, 0.6, 0), label='Tumor (pred/gt)'),
    ]

    for col, sl in enumerate(slices):
        ct_sl   = ct[:, :, sl]
        pred_sl = pred[:, :, sl]
        gt_sl   = gt[:, :, sl]

        axes[0, col].imshow(make_overlay(ct_sl, pred_sl).transpose(1, 0, 2), origin='lower')
        axes[0, col].set_title(f"Pred z={sl}", fontsize=9); axes[0, col].axis('off')

        axes[1, col].imshow(make_overlay(ct_sl, gt_sl).transpose(1, 0, 2), origin='lower')
        axes[1, col].set_title(f"GT z={sl}", fontsize=9); axes[1, col].axis('off')

    fig.legend(handles=legend_handles, loc='lower center', ncol=2,
               fontsize=10, frameon=True, bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout()
    save_path = os.path.join(WORKSPACE, f'overlay_{case_id}.png')
    plt.savefig(save_path, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved overlay → {save_path}")

## Cell 14 — Mosaic View: All Axial Slices for One Case

In [ ]:
import os, math
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

PRED_DIR  = os.path.join(WORKSPACE, 'predictions')
IMAGES_TS = os.path.join(RAW_DIR, 'Dataset003_Liver', 'imagesTs')

pred_files = sorted([f for f in os.listdir(PRED_DIR) if f.endswith('.nii.gz')])
case_file  = pred_files[0] if pred_files else None

if case_file is None:
    print("⚠️  No predictions found. Run Cell 8 first.")
else:
    case_id   = case_file.replace('_0000.nii.gz', '').replace('.nii.gz', '')
    ct_path   = os.path.join(IMAGES_TS, f"{case_id}_0000.nii.gz")
    pred_path = os.path.join(PRED_DIR, case_file)

    ct   = nib.load(ct_path).get_fdata().astype(np.float32)
    pred = nib.load(pred_path).get_fdata().astype(np.uint8)
    if ct.ndim   == 4: ct   = ct[0]
    if pred.ndim == 4: pred = pred[0]

    # Only slices with predicted liver or tumor
    active = [s for s in range(ct.shape[2]) if pred[:, :, s].max() > 0]
    if len(active) == 0:
        active = list(range(ct.shape[2]))

    n_cols = 8
    n_rows = math.ceil(len(active) / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 2.2, n_rows * 2.2))
    fig.suptitle(f'Mosaic — {case_id} (prediction overlay)', fontsize=13, fontweight='bold')
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for ax in axes:
        ax.axis('off')

    for i, sl in enumerate(active):
        ax = axes[i]
        ct_sl = ct[:, :, sl]
        ct_n  = (ct_sl - ct_sl.min()) / (ct_sl.max() - ct_sl.min() + 1e-8)
        rgb   = np.stack([ct_n, ct_n, ct_n], axis=-1).copy()

        p = pred[:, :, sl]
        rgb[p == 1, 2] = np.clip(rgb[p == 1, 2] + 0.45, 0, 1)  # blue liver
        rgb[p == 2, 1] = np.clip(rgb[p == 2, 1] + 0.55, 0, 1)  # green tumor

        ax.imshow(rgb.transpose(1, 0, 2), origin='lower')
        ax.set_title(f"z={sl}", fontsize=8)

    plt.tight_layout()
    save_path = os.path.join(WORKSPACE, f'mosaic_{case_id}.png')
    plt.savefig(save_path, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"✅ Mosaic saved → {save_path}")

## Cell 15 — Pipeline Summary & Next Steps

```
PIPELINE COMPLETED (Kaggle / NPY Dataset):
═══════════════════════════════════════════════════════════════
  ✅  Dataset       → zeynepzelk/task03-liver-npy-dataset
  ✅  Conversion    → NPY → NIfTI (liver+tumor masks merged)
  ✅  Preprocessing → Auto HU-clip, resample, normalize
  ✅  Training      → 2D config, fold 0, 100 epochs
  ✅  Inference     → Predictions on held-out validation cases
  ✅  Evaluation    → Dice, HD95, Jaccard (liver + tumor)
  ✅  Visualisation → Overlays, mosaics, metric bar & box plots
═══════════════════════════════════════════════════════════════
```

### To improve results:

| Action | Expected gain | How |
|---|---|---|
| Train all 5 folds | +1–2 Dice | Repeat Cell 7 with FOLD=1,2,3,4 |
| 1000 epochs | +2–3 Dice | Set `NUM_EPOCHS=1000` in Cell 7 |
| 3D_fullres config | +2–4 Dice liver | `CONFIGURATION='3d_fullres'`, use P100/A100 |
| ResEnc preset | +1–2 Dice | Add `-tr nnUNetTrainer_ResEncM` in Cell 7 |
| Ensemble all configs | +1–2 Dice | Run predict without `--disable_ensembling` |

### References
- nnUNet v2 GitHub: https://github.com/MIC-DKFZ/nnUNet  
- MSD dataset: http://medicaldecathlon.com  
- Kaggle dataset: https://www.kaggle.com/datasets/zeynepzelk/task03-liver-npy-dataset  
